# Обогащение как признаки ранкера: исследование и предобработка

Новые столбцы — сведения о компаниях из базы обогащения (`companies`, `company_facts`; выгрузка команды от 02.10.2026,
`scripts/db_merge_ydisk.sh`). Вопросы:

1. Какие столбцы вообще можно давать модели, не заглядывая в будущее относительно лота?
2. Есть ли сигнал сверх истории закупок?
3. Нет ли проблем, которые испортят обучение: утечка целевой переменной, информативные пропуски, дисбаланс, вырожденные признаки?

Результат — `data/enrichment_features/supplier_table.parquet` (одна строка на поставщика истории), из которого
`recsys/enrichment.py` считает признаки пар «лот × кандидат» на дату лота. Обучение и сравнение — `scripts/train_ranker.py`,
проверка на тестовом датасете 2026 — `scripts/eval_test2026.py`.

In [1]:
import json, os, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.append(str(ROOT / 'web-service'))
from recsys.data import BASE_YEAR
from recsys.enrichment import build_supplier_table, pair_features, ENRICH_SPEC, ENRICH_SAME_YEAR, MIN_FIN_YEAR
OUT = ROOT / 'data' / 'enrichment_features'; OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 200, 'display.max_columns', 30, 'display.max_colwidth', 80)

## 1. Выгрузка из базы

Берём факты по полям-кандидатам со всеми источниками (выбор значения по приоритету источников — как в карточке
`enrichment/card.py`). Из базы, а не из `companies_enriched.csv`: в CSV финансы только за последний год, а для честных
признаков нужна отчётность по годам (`finance_by_year`).

In [2]:
import psycopg2
from integrations.new_pool import db_url
FIELDS = ['finance_by_year', 'employees', 'employees_rmsp', 'employees_as_of', 'smp_as_of', 'smp_since', 'smp_category',
          'reg_date', 'taxes_paid', 'taxes_paid_as_of', 'okved_main', 'okved_extra', 'in_gisp', 'in_software_registry',
          'licenses_count', 'ogrn', 'gisp_okpd2', 'software_okpd2']
with psycopg2.connect(db_url()) as conn:
    facts = pd.read_sql('select inn, field, source, value::text as value, fetched_at from company_facts where field = any(%s)', conn, params=(FIELDS,))
    companies = pd.read_sql('select inn, kind, region_code, is_smp, status, is_active from companies', conn)
facts.to_parquet(OUT / 'facts_raw.parquet'); companies.to_parquet(OUT / 'companies_raw.parquet')
suppliers = pd.read_parquet(ROOT / 'models' / 'suppliers.parquet')[['sid', 'inn']]   # поставщики истории, sid как в модели
print(f'фактов {len(facts):,}, компаний в базе {len(companies):,}, поставщиков в истории {len(suppliers):,}, '
      f'из них в базе {suppliers.inn.isin(companies.inn).mean():.1%}')

фактов 401,115, компаний в базе 45,560, поставщиков в истории 44,174, из них в базе 100.0%


## 2. Даты сведений: что известно на момент лота

Обучающие лоты — 2024–2025, тестовые — октябрь–декабрь 2025 и тестовый датасет организаторов (январь–август 2026).

In [3]:
asof = {f: facts.loc[facts.field == f, 'value'].str[1:11].value_counts().to_dict() for f in ['smp_as_of', 'employees_as_of', 'taxes_paid_as_of']}
years = facts.loc[facts.field == 'finance_by_year', 'value'].map(lambda s: tuple(sorted(json.loads(s)))).value_counts().head(6)
print('Дата сведений:', json.dumps(asof, ensure_ascii=False)); print('Годы отчётности (ГИР БО) у компаний:'); years

Дата сведений: {"smp_as_of": {"2026-09-10": 26378}, "employees_as_of": {"2025-12-31": 18022}, "taxes_paid_as_of": {"2025-12-31": 18552}}
Годы отчётности (ГИР БО) у компаний:


value
(2021, 2022, 2023, 2024, 2025)    15427
(2022, 2023, 2024, 2025)            888
(2023, 2024, 2025)                  861
(2024, 2025)                        700
(2025,)                             460
(2021, 2022, 2023, 2024)            387
Name: count, dtype: int64

- **Численность и налоги** — один срез, за 2025 год. Для лотов 2025 года это тот же год (возможна утечка: налоги
  2025 года включают оплату тех самых контрактов), для лотов 2026 года — прошлый год.
- **Реестр МСП** — снимок 10.09.2026: категория и дата включения (`smp_since`) — по дате включения видно, был ли
  поставщик МСП на дату лота.
- **Финансы** — по годам 2021–2025. Берём **год, предшествующий году лота** (лоты 2025 → отчётность 2024, лоты 2026 → 2025).
  Отчётность раньше 2024 года не используем совсем: на 2026 год она устарела (это же правило в карточке компании,
  `enrichment/card.py: drop_stale`). Поэтому у лотов 2024 года финансов нет.

**Не берём вообще** (до любых расчётов):

| Столбцы | Почему |
|---|---|
| `hist_*`, `role`, `role_label` | посчитаны по той же истории закупок, что и метки, включая тестовые месяцы — прямая утечка |
| `contact_*`, `contacts_found` | контакты ищутся по номерам **выигранных** закупок — пропуск зависит от победы |
| `in_rnp`, `rnp_ever`, `risk_flags` | записи РНП появляются после срыва контрактов, в т. ч. выигранных в обучающем периоде |
| `status`, `is_active`, `liquidation_date` | состояние на 2026 год: «выжившие» компании чаще побеждали — смещение выживших |
| `revenue`, `taxes_paid` … из `companies` | «последний год» без привязки к дате лота |

## 3. Предобработка: таблица поставщиков

In [4]:
E = suppliers.merge(build_supplier_table(facts, companies), on='inn', how='left')
E.to_parquet(OUT / 'supplier_table.parquet')
cov = E.drop(columns=['sid', 'inn', 'okved_groups', 'okved_classes']).notna().mean().rename('покрытие').to_frame()
cov['пример'] = [E[c].dropna().iloc[0] if E[c].notna().any() else None for c in cov.index]
cov.round(3)

,покрытие,пример
reg_month,0.598,-42.0
smp_since_month,0.597,-46.0
msp_cat,0.597,1.0
in_msp_snapshot,1.000,0
revenue_2024,0.409,2211899000.0
net_profit_2024,0.411,52700000.0
assets_2024,0.422,737919000.0
equity_2024,0.405,561682000.0
revenue_2025,0.410,2111499000.0
net_profit_2025,0.411,28342000.0


Преобразования (`recsys/enrichment.py`):

- **возраст на дату лота** — по дате регистрации, иначе по году из ОГРН (середина года); отрицательный — пропуск;
- **категория МСП на дату лота** — только если дата включения в реестр не позже месяца лота;
- **деньги** — `log(1+x)`, выручка/прибыль/активы/капитал за год до лота; рентабельность обрезана в [−1, 1];
  отношение выручки к НМЦК — разность логарифмов;
- **ОКВЭД ↔ ОКПД2 лота** — прямое совпадение кода XX.XX с основной группой лота (основной ОКВЭД / любой) и класса XX.
  Выученное соответствие ОКВЭД→ОКПД2 не используем: оно построено по победителям всех месяцев, включая тестовые;
- **пропуски не заполняем**: LightGBM учится отдельно на пропуске, а «нет данных» ≠ «ноль».

## 4. Пары «лот × кандидат»: покрытие, сигнал, утечка

Берём кандидатов из сохранённых признаков ранкера (`data/features_v2`): обучающие месяцы апрель–август 2025 (выборка 60+30
отрицательных на лот, как при обучении) и тестовые октябрь–декабрь 2025 (все кандидаты). По 3 000 лотов с победителем
среди кандидатов на месяц.

**Сила признака** — средняя по лотам AUC: вероятность, что у победителя значение выше, чем у случайного другого
кандидата того же лота (0,5 — нет сигнала). Именно так признак работает в ранжировании — внутри лота.

In [5]:
lots = pd.read_parquet(ROOT / 'data/processed/lots.parquet', columns=['lot_id', 'publish_date', 'main_group', 'is_smp'])
d = pd.to_datetime(lots.publish_date); lots['month'] = (d.dt.year - BASE_YEAR) * 12 + d.dt.month - 1
rng = np.random.default_rng(0)

def load(folder, n_lots=3000):
    parts = sorted((ROOT / 'data/features_v2' / folder).glob('part*.parquet'))
    F = pd.concat([pd.read_parquet(p, columns=['lot_id', 'sid', 'label', 'lot_price_log', 's_n_win', 'code_win_l3', 'cust_win']) for p in parts])
    keep = rng.choice(F.loc[F.label == 2, 'lot_id'].unique(), n_lots, replace=False)
    F = F[F.lot_id.isin(keep)].merge(lots, on='lot_id').reset_index(drop=True)
    X = pair_features(E, F[['sid', 'month', 'main_group']].assign(price_log=F.lot_price_log.values))
    x = F[['sid']].merge(E[['sid', 'revenue_2024', 'revenue_2025']], on='sid', how='left')
    X['diag_rev_2024'] = np.log1p(x.revenue_2024.clip(lower=0)).values   # диагностика утечки: тот же признак за 2024
    X['diag_rev_2025'] = np.log1p(x.revenue_2025.clip(lower=0)).values   # … и за 2025 — год самих лотов
    return pd.concat([F, X], axis=1).assign(part='test' if folder in ('m21', 'm22', 'm23') else 'train')

P = pd.concat([load(f'm{m}_full') for m in range(15, 20)] + [load(f'm{m}') for m in (21, 22, 23)], ignore_index=True)
print(f'пар {len(P):,}, лотов {P.lot_id.nunique():,}')

пар 4,030,069, лотов 24,000


### 4.1 Баланс классов

In [6]:
bal = P.groupby('part').agg(пар=('label', 'size'), лотов=('lot_id', 'nunique'),
                           победителей=('label', lambda s: (s == 2).sum()), участников_без_победы=('label', lambda s: (s == 1).sum()))
bal['кандидатов_на_лот'] = bal.пар / bal.лотов
bal['доля_победителей'] = bal.победителей / bal.пар
bal.round(4)

,пар,лотов,победителей,участников_без_победы,кандидатов_на_лот,доля_победителей
part,,,,,,
test,2654168,9000,9004,6771,294.9076,0.0034
train,1375901,15000,15002,10899,91.7267,0.0109


Победитель — меньше 1–2% пар, но для LambdaRank важен баланс **внутри лота**: в каждой группе ровно один
(редко несколько) победитель среди ~90 (обучение) или ~250 (тест) кандидатов. Пересэмплирование не нужно и даже вредно:
оно меняет состав кандидатов, на котором считается NDCG. Лоты без победителя среди кандидатов (~17%) в обучение не
попадают — это ограничение генератора кандидатов, а не модели. Новые признаки баланс не меняют.

### 4.2 Покрытие и сила признаков

In [7]:
FEATS = [f[0] for f in ENRICH_SPEC] + [f[0] for f in ENRICH_SAME_YEAR] + ['diag_rev_2024', 'diag_rev_2025']
BASE = ['s_n_win', 'code_win_l3', 'cust_win']

def lot_auc(df, f):
    d = df[['lot_id', 'label', f]].dropna()
    d = d.assign(pos=(d.label == 2).astype(int), r=d.groupby('lot_id')[f].rank())
    g = d.groupby('lot_id').agg(n=('pos', 'size'), npos=('pos', 'sum'))
    g['rpos'] = d[d.pos == 1].groupby('lot_id').r.sum()
    g = g[(g.npos > 0) & (g.n > g.npos)]
    return ((g.rpos - g.npos * (g.npos + 1) / 2) / (g.npos * (g.n - g.npos))).mean()

rows = []
for f in FEATS + BASE:
    for part, df in P.groupby('part'):
        rows.append({'признак': f, 'часть': part, 'AUC в лоте': lot_auc(df, f),
                     'покрытие: победители': df.loc[df.label == 2, f].notna().mean(),
                     'покрытие: остальные': df.loc[df.label < 2, f].notna().mean()})
R = pd.DataFrame(rows).pivot(index='признак', columns='часть').reindex(FEATS + BASE)
R.round(3)

AUC в лоте        покрытие: победители        покрытие: остальные       
часть                 test  train                 test  train                test  train
признак                                                                                 
e_age_years          0.539  0.512                0.813  0.796               0.812  0.793
e_msp_cat            0.525  0.519                0.752  0.756               0.785  0.763
e_rev_log            0.574  0.553                0.663  0.641               0.619  0.630
e_margin             0.520  0.510                0.661  0.638               0.615  0.626
e_assets_log         0.562  0.552                0.671  0.651               0.629  0.638
e_neg_equity         0.494  0.491                0.648  0.626               0.607  0.616
e_rev_to_price       0.574  0.553                0.663  0.641               0.619  0.630
e_okved_main         0.541  0.522                0.756  0.756               0.808  0.787
e_okved_any          0.578  0.552                0.756  0.756               0.808  0.787
e_okved_class        0.582  0.549                0.756  0.756               0.808  0.787
e_n_okved            0.509  0.491                0.756  0.757               0.808  0.787
e_gisp               0.499  0.502                1.000  1.000               1.000  1.000
e_software           0.513  0.501                1.000  1.000               1.000  1.000
e_licenses           0.526  0.517                0.758  0.765               0.795  0.778
e_employees_log      0.587  0.567                0.641  0.618               0.603  0.605
e_taxes_log          0.588  0.566                0.642  0.619               0.612  0.615
diag_rev_2024        0.574  0.553                0.663  0.641               0.619  0.630
diag_rev_2025        0.596  0.576                0.691  0.653               0.617  0.622
s_n_win              0.697  0.647                1.000  1.000               1.000  1.000
code_win_l3          0.826  0.746                1.000  1.000               1.000  1.000
cust_win             0.718  0.656                1.000  1.000               1.000  1.000

### 4.3 Утечка: один и тот же показатель за разные годы

`diag_rev_2024` и `diag_rev_2025` — выручка одной и той же компании на одних и тех же лотах 2025 года. Если бы выручка
просто описывала размер компании, сигнал был бы одинаковым. Выручка 2025 года сильнее и у победителей чаще заполнена —
в неё входят деньги по этим самым контрактам. Численность и налоги есть только за 2025 год и ведут себя так же,
как выручка 2025. **Поэтому в модель идут только финансы за год до лота, а численность и налоги — нет**
(для лотов 2026 года они были бы честными, но учить модель на них не на чем: на обучающих лотах это утечка).

In [8]:
R.loc[['diag_rev_2024', 'diag_rev_2025', 'e_rev_log', 'e_employees_log', 'e_taxes_log']].round(3)

AUC в лоте        покрытие: победители        покрытие: остальные       
часть                 test  train                 test  train                test  train
признак                                                                                 
diag_rev_2024        0.574  0.553                0.663  0.641               0.619  0.630
diag_rev_2025        0.596  0.576                0.691  0.653               0.617  0.622
e_rev_log            0.574  0.553                0.663  0.641               0.619  0.630
e_employees_log      0.587  0.567                0.641  0.618               0.603  0.605
e_taxes_log          0.588  0.566                0.642  0.619               0.612  0.615

### 4.4 Вырожденные и редкие признаки, взаимодействие с типом закупки

In [9]:

bin_cols = ['e_okved_main', 'e_okved_any', 'e_okved_class', 'e_gisp', 'e_software', 'e_neg_equity']
tab = pd.DataFrame({c: {'доля 1 среди кандидатов': P[c].mean(),
                        'у победителей (МСП-лоты)': P.loc[P.is_smp & (P.label == 2), c].mean(),
                        'у остальных (МСП-лоты)': P.loc[P.is_smp & (P.label < 2), c].mean(),
                        'у победителей (прочие)': P.loc[~P.is_smp & (P.label == 2), c].mean(),
                        'у остальных (прочие)': P.loc[~P.is_smp & (P.label < 2), c].mean()} for c in bin_cols}).T
tab.round(3)

,доля 1 среди кандидатов,у победителей (МСП-лоты),у остальных (МСП-лоты),у победителей (прочие),у остальных (прочие)
e_okved_main,0.075,0.124,0.073,0.133,0.075
e_okved_any,0.221,0.343,0.225,0.340,0.219
e_okved_class,0.289,0.408,0.295,0.404,0.286
e_gisp,0.016,0.022,0.019,0.017,0.015
e_software,0.022,0.010,0.018,0.037,0.023
e_neg_equity,0.050,0.024,0.046,0.036,0.051


Признак «субъект МСП на дату лота» оказался вырожденным: значение известно только для компаний из **текущего**
реестра МСП, и у них почти всегда 1 (98,5%) — он лишь повторяет пропуск. Из модели убран, оставлена категория МСП.

### 4.5 Дублирует ли обогащение историю закупок

In [10]:
P[[f[0] for f in ENRICH_SPEC] + ['s_n_win', 'code_win_l3']].rank().corr(method='pearson').loc[[f[0] for f in ENRICH_SPEC], ['s_n_win', 'code_win_l3']].round(2)

,s_n_win,code_win_l3
e_age_years,0.09,0.03
e_msp_cat,0.04,0.07
e_rev_log,0.17,0.15
e_margin,0.07,0.01
e_assets_log,0.16,0.13
e_neg_equity,-0.05,-0.02
e_rev_to_price,0.15,0.03
e_okved_main,-0.10,0.16
e_okved_any,-0.11,0.24
e_okved_class,-0.14,0.20


## 5. Отбор признаков в модели

Одномерная сила не учитывает взаимодействия (реестр ПО важен только для лотов с ПО), поэтому окончательный отбор — по
обученной модели v3 (49 признаков v2 + 14 новых): **важность перестановкой на валидации** (сентябрь 2025, 5 000 лотов,
`scripts/perm_importance.py`). Значения признака перемешиваются между кандидатами внутри лота, 3 повтора; смотрим падение
NDCG@10. Тест (октябрь–декабрь) в отборе не участвует.

In [11]:
perm = pd.read_csv(ROOT / 'data/features_v2/perm_v3_full.csv')
perm['ΔNDCG@10, п.'] = (perm.drop_mean * 100).round(3); perm['± п.'] = (perm.drop_std * 100).round(3)
perm['доля gain'] = perm.gain_share.round(4)
perm[['feature', 'ΔNDCG@10, п.', '± п.', 'доля gain']]

,feature,"ΔNDCG@10, п.",± п.,доля gain
0,e_n_okved,0.203,0.019,0.0058
1,e_age_years,0.185,0.115,0.0094
2,e_assets_log,0.163,0.053,0.0056
3,e_margin,0.124,0.040,0.0052
4,e_msp_cat,0.098,0.053,0.0025
5,e_rev_log,0.090,0.047,0.0083
6,e_licenses,0.073,0.082,0.0025
7,e_okved_any,0.050,0.012,0.0008
8,e_rev_to_price,0.043,0.030,0.0027
9,e_software,0.020,0.009,0.0003


Без `e_okved_main`, `e_okved_class`, `e_gisp`, `e_neg_equity` NDCG@10 не падает (падение в пределах разброса):
основной ОКВЭД и класс дублируют `e_okved_any`, ГИСП и отрицательный капитал — шум. Вредными они не являются (падение не
отрицательное за пределами разброса), поэтому в сервис идёт **v3 со всеми 14 признаками** — именно она проверена на тесте.
Облегчённый набор (`FEATURES_V3L` в `recsys/features.py`, 59 признаков) — следующий шаг: переобучить и сравнить по валидации.

Финальная модель сервиса (`models/`) — v3, переобученная на всех месяцах апрель–декабрь 2025 с тем же числом итераций
(`train_ranker.py save v3_full --refit`): метрики ниже — с честного разбиения (обучение апрель–август, тест октябрь–декабрь).

In [12]:
res = json.loads((ROOT / 'data/features_v2/results.json').read_text(encoding='utf-8'))
names = {'prod_v1': 'рабочая модель (v1, 33 признака)', 'v2_full': 'v2: история + свежесть (49)',
         'v3_full': 'v3: + обогащение (63)', 'v3l_full': 'v3l: + обогащение после отбора (59)'}
rows = []
for k, title in names.items():
    if k not in res: continue
    t = res[k]['test']
    rows.append({'модель': title, 'Recall@1': t['Все']['Recall@1'], 'Recall@5': t['Все']['Recall@5'], 'Recall@10': t['Все']['Recall@10'],
                 'MRR': t['Все']['MRR'], 'NDCG@10': t['Все']['NDCG@10'], 'новый для заказчика R@10': t['2. новый для заказчика']['Recall@10']})
pd.DataFrame(rows).set_index('модель').round(3)

,Recall@1,Recall@5,Recall@10,MRR,NDCG@10,новый для заказчика R@10
модель,,,,,,
"рабочая модель (v1, 33 признака)",0.377,0.608,0.693,0.484,0.536,0.519
v2: история + свежесть (49),0.382,0.615,0.700,0.489,0.541,0.537
v3: + обогащение (63),0.387,0.620,0.704,0.494,0.547,0.547


## 6. Выводы

- **Признаки в модели (14):** возраст и категория МСП на дату лота; выручка, рентабельность, активы, отрицательный капитал
  и выручка относительно НМЦК за год до лота; совпадение ОКВЭД с группой/классом ОКПД2 лота и число ОКВЭД; реестры ГИСП
  и ПО; число лицензий. По перестановке полезнее всего число ОКВЭД, возраст, активы, рентабельность, категория МСП, выручка.
- **Не в модели:** численность и налоги за 2025 (утечка на обучающих лотах), история/роль/контакты/РНП/статус
  (утечка целевой переменной или сведения из будущего), «МСП на дату лота» (вырожденный).
- **Сила:** по отдельности признаки слабые (AUC в лоте 0,51–0,58 против 0,65–0,83 у истории), зато слабо связаны с
  историей (|ρ| ≤ 0,24) и вместе дают прирост: тест октябрь–декабрь 2025 — Recall@10 0,693 → 0,704 к рабочей модели,
  в сегменте «новый для заказчика» 0,519 → 0,547, где истории мало.
- **Пропуски информативны умеренно:** у победителей финансы заполнены на 1–4 п. чаще (крупные компании чаще сдают
  отчётность в ГИР БО), это законный сигнал размера, а не утечка — отчётность за прошлый год сдана до лота.
- **Дисбаланс** — внутри лота 1 победитель на ~90–250 кандидатов; LambdaRank с группами по лотам рассчитан на это,
  пересэмплирование не нужно. Новые признаки распределение классов не меняют.